## TASK 1: Data Understanding

In [1]:
import pandas as pd

df = pd.read_csv("IMDB_Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [2]:
df.shape
df['sentiment'].value_counts()

sentiment
positive    25000
negative    25000
Name: count, dtype: int64

In [3]:
df.shape
df['sentiment'].value_counts()

sentiment
positive    25000
negative    25000
Name: count, dtype: int64

## TASK 2: NLP Preprocessing

In [6]:
import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to C:\Users\K S
[nltk_data]     V\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package punkt to C:\Users\K S
[nltk_data]     V\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package wordnet to C:\Users\K S
[nltk_data]     V\AppData\Roaming\nltk_data...


True

>## Step-by-step preprocessing

In [7]:
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-zA-Z]", " ", text)
    
    words = text.split()
    words = [w for w in words if w not in stop_words]
    words = [lemmatizer.lemmatize(w) for w in words]
    
    return " ".join(words)

>## Apply preprocessing

In [8]:
df['cleaned_review'] = df['review'].apply(preprocess_text)

## TASK 3: Feature Engineering
>## 1. Bag of Words

In [9]:
from sklearn.feature_extraction.text import CountVectorizer
bow = CountVectorizer(max_features=5000)
X_bow = bow.fit_transform(df['cleaned_review']).toarray()

>## 2. TF-IDF

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf.fit_transform(df['cleaned_review']).toarray()

>## Convert labels

In [11]:
y = df['sentiment'].map({'positive':1, 'negative':0})

## TASK 4: Model Building

In [12]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42
)

>## Model 1: Logistic Regression

In [13]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression()
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

>## Model 2: Naive Bayes

In [14]:
from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()
nb.fit(X_train, y_train)

y_pred_nb = nb.predict(X_test)

>## Model 3: Decision Tree

In [15]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)

## TASK 5: Model Evaluation

In [16]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

>## Create evaluation function

In [17]:
def evaluate(y_test, y_pred, model_name):
    print(f"{model_name}")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall:", recall_score(y_test, y_pred))
    print("F1 Score:", f1_score(y_test, y_pred))
    print("-"*40)

>## Evaluate all models

In [18]:
evaluate(y_test, y_pred_lr, "Logistic Regression")
evaluate(y_test, y_pred_nb, "Naive Bayes")
evaluate(y_test, y_pred_dt, "Decision Tree")

Logistic Regression
Accuracy: 0.8883
Precision: 0.8793036750483559
Recall: 0.9021631276046834
F1 Score: 0.8905867371926731
----------------------------------------
Naive Bayes
Accuracy: 0.8561
Precision: 0.8528028224225794
Recall: 0.8634649732089701
F1 Score: 0.8581007790158761
----------------------------------------
Decision Tree
Accuracy: 0.7134
Precision: 0.7159610415424369
Recall: 0.7148243699146656
F1 Score: 0.7153922542204568
----------------------------------------


## TASK 6: Comparison & Insights

>### Example:
- Logistic Regression → best accuracy
- Naive Bayes → fast but slightly lower accuracy
- Decision Tree → overfitting observed

>### Also mention:
- TF-IDF performed better than BoW
- Lemmatization improved results